In [0]:
# Importando o current_timestamp e o 'col' (para referenciar a coluna oculta)
from pyspark.sql.functions import current_timestamp, col

# 1. Definindo o caminho onde os arquivos estão salvos no seu Volume
caminho_distribuidoras = "/Volumes/projeto_ancine/bronze/raw_ancine/bilheteria_diaria_obras_por_distribuidoras/*.csv"

# 2. Lendo todos os CSVs de uma vez, mantendo tudo como texto (String)
df_csv = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "false")
    .load(caminho_distribuidoras)
)

# 3. Adicionando metadados de auditoria usando o padrão Unity Catalog
df_bronze = (
    df_csv
    .withColumn("arquivo_origem", col("_metadata.file_path")) 
    .withColumn("data_ingestao", current_timestamp())
)

# 4. Salvando como Tabela Delta no seu schema 'bronze'
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_ancine.bronze.bilheteria_distribuidoras")
)

print("Ingestão da Bronze (Distribuidoras) concluída com sucesso via Unity Catalog!")



In [0]:
# 1. Definindo o caminho para a pasta das exibidoras (ajuste se o nome da sua pasta estiver diferente)
caminho_exibidoras = "/Volumes/projeto_ancine/bronze/raw_ancine/bilheteria_diaria_obras_por_exibidoras/*.csv"

# 2. Leitura bruta de todos os 664 CSVs
df_csv_exibidoras = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "false")
    .load(caminho_exibidoras)
)

# 3. Adicionando metadados de auditoria (Unity Catalog)
df_bronze_exibidoras = (
    df_csv_exibidoras
    .withColumn("arquivo_origem", col("_metadata.file_path"))
    .withColumn("data_ingestao", current_timestamp())
)

# 4. Salvando como a segunda Tabela Delta da nossa Camada Bronze
(
    df_bronze_exibidoras.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_ancine.bronze.bilheteria_exibidoras")
)

print("Ingestão da Bronze (Exibidoras) concluída com sucesso via Unity Catalog!")

In [0]:
# 1. Definindo o caminho para o arquivo de lançamentos comerciais
caminho_lancamentos = "/Volumes/projeto_ancine/bronze/raw_ancine/lancamentos_comerciais_por_distribuidoras/*.csv"

# 2. Leitura bruta do CSV
df_csv_lancamentos = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "false")
    .load(caminho_lancamentos)
)

# 3. Adicionando metadados de auditoria (Unity Catalog)
df_bronze_lancamentos = (
    df_csv_lancamentos
    .withColumn("arquivo_origem", col("_metadata.file_path"))
    .withColumn("data_ingestao", current_timestamp())
)

# 4. Salvando como a terceira Tabela Delta da nossa Camada Bronze
(
    df_bronze_lancamentos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_ancine.bronze.lancamentos_comerciais")
)

print("Ingestão da Bronze (Lançamentos Comerciais) concluída com sucesso via Unity Catalog!")

In [0]:
display(spark.read.table("projeto_ancine.bronze.bilheteria_distribuidoras").limit(5))